In [1]:
import pandas as pd

In [48]:
import pandas as pd
from pathlib import Path

videopath = 'GX011264.MP4'
folder = Path(rf"C:\Users\uriel\Documents\OneDrive\Escritorio\pothole\potholeGDL\runs\detect\video_tracking_custom\{videopath}\labels")

dfs = []
for f in folder.glob("*.txt"):
    df = pd.read_csv(
        f,
        sep=" ",
        header=None,
        names=["class", "x_center", "y_center", "width", "height"]
    )
    df["filename"] = f.name
    dfs.append(df)

df_all = pd.concat(dfs, ignore_index=True)
print(df_all)


     class  x_center  y_center     width    height  \
0        2  0.485234  0.808462  0.206396  0.361600   
1        2  0.465194  0.821731  0.205173  0.334769   
2        2  0.444842  0.833993  0.241538  0.303442   
3        3  0.756163  0.904626  0.068979  0.101911   
4        2  0.594273  0.768239  0.338929  0.416601   
..     ...       ...       ...       ...       ...   
792      2  0.788904  0.500000  0.422192  1.000000   
793      2  0.823205  0.500000  0.353590  1.000000   
794      3  0.231190  0.716918  0.050324  0.055753   
795      3  0.284254  0.756610  0.053436  0.057909   
796      3  0.343956  0.817474  0.059065  0.065046   

                          filename  
0    GX011264.MP4_frame_000001.txt  
1    GX011264.MP4_frame_000002.txt  
2    GX011264.MP4_frame_000003.txt  
3    GX011264.MP4_frame_000004.txt  
4    GX011264.MP4_frame_000007.txt  
..                             ...  
792  GX011264.MP4_frame_002852.txt  
793  GX011264.MP4_frame_002853.txt  
794  GX011264.MP4_

In [49]:
average = df_all["class"].value_counts().mean().round(0)
df_all["class"].value_counts()

class
2    535
0    136
3     90
1     36
Name: count, dtype: int64

In [50]:
import shutil
import os

src = Path(r"C:\Users\uriel\Documents\OneDrive\Escritorio\pothole\potholeGDL\runs\detect\video_tracking_custom")
dst = Path(r"C:\Users\uriel\Documents\OneDrive\Escritorio\pothole\potholeGDL\newdataset")

dst.mkdir(exist_ok=True)

for _, row in df_all.iterrows():
    filename = row["filename"]
    src_file = os.path.join(src, videopath, "labels", filename)
    dst_file = os.path.join(dst, 'labels', filename)

    src_image = os.path.join(src, videopath, "images", filename.replace(".txt", ".jpg"))
    dst_image = os.path.join(dst, 'images', filename.replace(".txt", ".jpg"))

    # Copiar archivo original
    if src_file:
        shutil.copy(src_file, dst_file)
    # Copiar archivo original
    if src_image:
        shutil.copy(src_image, dst_image)


In [51]:
import random
from pathlib import Path
import shutil

# Folder de imágenes
img_folder = Path(os.path.join(src, videopath, "images"))

# Folder de labels existentes

# Nuevo folder donde copiarás imágenes y txt vacíos
dst_img = Path(os.path.join(dst, 'images'))
dst_lbl = Path(os.path.join(dst, 'labels'))

dst_img.mkdir(exist_ok=True)
dst_lbl.mkdir(exist_ok=True)

# 1. Lista de labels existentes (sin extensión)
label_files = {f.stem for f in folder.glob("*.txt")}

# 2. Lista de imágenes
image_files = list(img_folder.glob("*.jpg")) + list(img_folder.glob("*.png"))

# 3. Filtrar imágenes que NO tienen detecciones
images_without_labels = [
    img for img in image_files
    if img.stem not in label_files
]

print(f"Total imágenes sin detecciones: {len(images_without_labels)}")

# 4. Seleccionar 400 aleatorias
selected = random.sample(images_without_labels, int(average))

# 5. Copiar imágenes y crear txt vacío
for img in selected:
    # Copiar imagen
    shutil.copy(img, dst_img / img.name)

    # Crear txt vacío
    empty_txt = dst_lbl / f"{img.stem}.txt"
    empty_txt.write_text("")   # archivo vacío

Total imágenes sin detecciones: 2183
